# **Evaluación 2 de Inteligencia Artificial**
# **Modelos de Regresión**


Estudiantes ***Grupo 4***:
* Cristina Morán cristina.moran2201@alumnos.ubiobio.cl
* Alonso Valderrama alonso.valderrama2201@alumnos.ubiobio.cl
* Amasis Guzmán amasis.guzman2201@alumnos.ubiobio.cl

Profesora Jazna Meza Hidalgo

Ingeniería Civil en Informatica - UBB


# **Importación librerías**

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, confusion_matrix
from sklearn.datasets import make_classification

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer, StandardScaler

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

# **Carga de Datos**

In [4]:
df = pd.read_csv('data/Ingestion/retail_ventas.csv')
print(f'Dimensiones del dataset: {df.shape}')
df.head()

Dimensiones del dataset: (25500, 11)


,id_tienda,tamano_tienda,antiguedad_tienda,temperatura,precio_combustible,indice_economico,semana,es_feriado,clientes_estimados,tipo_tienda,ventas_semanales
0,86,1445.454837,1,2.611389,0.925557,97.313562,41,0,376.608869,B,40377.989406
1,35,1973.195744,18,5.603718,1.526210,88.821903,52,0,197.329419,A,59430.330077
2,41,1727.496264,24,7.330847,1.297304,109.273193,15,0,262.737637,A,56547.323925
3,97,1284.634121,10,16.074751,0.862449,98.362477,45,0,392.171919,A,36408.752020
4,25,1457.596546,5,9.278104,1.264497,92.081812,45,0,411.085409,B,40500.513890


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25500 entries, 0 to 25499
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_tienda           25500 non-null  int64  
 1   tamano_tienda       25500 non-null  float64
 2   antiguedad_tienda   25500 non-null  int64  
 3   temperatura         24224 non-null  float64
 4   precio_combustible  24221 non-null  float64
 5   indice_economico    25500 non-null  float64
 6   semana              25500 non-null  int64  
 7   es_feriado          25500 non-null  int64  
 8   clientes_estimados  24220 non-null  float64
 9   tipo_tienda         25500 non-null  object 
 10  ventas_semanales    25500 non-null  float64
dtypes: float64(6), int64(4), object(1)
memory usage: 2.1+ MB


In [6]:
df.describe()

,id_tienda,tamano_tienda,antiguedad_tienda,temperatura,precio_combustible,indice_economico,semana,es_feriado,clientes_estimados,ventas_semanales
count,25500.00000,25500.000000,25500.000000,24224.000000,24221.000000,25500.000000,25500.000000,25500.000000,24220.000000,25500.000000
mean,50.81749,1501.338719,15.092824,14.954302,1.303299,99.889523,26.376078,0.098510,300.649372,43108.586339
std,28.93800,399.711835,8.365652,9.999174,0.299869,9.961593,14.996831,0.298009,98.822402,14538.319334
min,1.00000,500.000000,1.000000,-24.344608,0.086903,60.799835,1.000000,0.000000,50.000000,4905.208771
25%,26.00000,1229.224069,8.000000,8.124612,1.102983,93.140607,13.000000,0.000000,234.014040,33827.197346
50%,51.00000,1503.366822,15.000000,14.930043,1.303386,99.917526,26.000000,0.000000,301.600830,42117.538361
75%,76.00000,1773.575577,22.000000,21.735814,1.505762,106.582946,39.000000,0.000000,366.981593,50947.405475
max,100.00000,2901.612393,29.000000,56.005247,2.455792,139.836638,52.000000,1.000000,680.907511,234175.036973


**Revisión de valores nulos**

In [7]:
print('Valores nulos por columna:')
print(df.isnull().sum())

Valores nulos por columna:
id_tienda                0
tamano_tienda            0
antiguedad_tienda        0
temperatura           1276
precio_combustible    1279
indice_economico         0
semana                   0
es_feriado               0
clientes_estimados    1280
tipo_tienda              0
ventas_semanales         0
dtype: int64


**Verificar porcentaje de valores nulos en las columnas correspondientes**

In [8]:
columnas_con_nulos = df.isna().sum()[df.isna().sum() > 0]
porcentaje_nulos = (columnas_con_nulos / df.shape[0]) * 100

resultado = pd.DataFrame({
    "Cantidad Nulos": columnas_con_nulos,
    "Porcentaje Nulos (%)": porcentaje_nulos
}).round(2)

resultado

,Cantidad Nulos,Porcentaje Nulos (%)
temperatura,1276,5.00
precio_combustible,1279,5.02
clientes_estimados,1280,5.02


**Revisión y eliminación de valores duplicados**

In [9]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {duplicados}')

if duplicados > 0:
    df = df.drop_duplicates()
    print(f'Filas tras eliminar duplicados: {df.shape[0]}')
else:
    print('No se encontraron filas duplicadas.')

Filas duplicadas encontradas: 494
Filas tras eliminar duplicados: 25006


# **Preparación de datos**

**Separación variables predictoras y variable objetivo**

In [10]:
# Variable objetivo
y = df['ventas_semanales']

# Variables predictoras
X = df.drop(columns=['id_tienda','ventas_semanales'])

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

Shape de X: (25006, 9)
Shape de y: (25006,)


**Variables numéricas y categóricas**


In [11]:
# Variables
features_num = [
    'tamano_tienda', 'antiguedad_tienda', 'temperatura','precio_combustible',
    'indice_economico', 'semana', 'clientes_estimados'
]
features_cat = ['tipo_tienda']

print('Variables numéricas:', features_num)
print('Variables categóricas:', features_cat)


Variables numéricas: ['tamano_tienda', 'antiguedad_tienda', 'temperatura', 'precio_combustible', 'indice_economico', 'semana', 'clientes_estimados']
Variables categóricas: ['tipo_tienda']


Se excluye el id_tienda ya que es un identificador y tampoco aporta infromación predictiva

# **Modelamiento**

In [12]:
# Eliminamos duplicados antes del pipeline
df = df.drop_duplicates()
print(f'Shape después de eliminar duplicados: {df.shape}')

Shape después de eliminar duplicados: (25006, 11)


**Pipeline de preprocesamiento**

In [13]:
# Preprocesamiento numérico: imputación con mediana
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

# Preprocesamiento categórico: imputación + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat)
    ],
    remainder="passthrough",
    force_int_remainder_cols=False
)

**División datos (Train/Test Split)**

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Regresión lineal múltiple**

In [16]:
pipeline_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

In [17]:
# Entrenamiento
pipeline_lr.fit(X_train, y_train)

# Predicciones
y_pred_lr = pipeline_lr.predict(X_test)

# **Ridge**

In [18]:
pipeline_ridge = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(random_state=42))
    ]
)

In [19]:
# Probar 4 valores ditintos de alpha
parametros_ridge = {
    "regressor__alpha": [0.1, 1.0, 10.0, 100.0]
}

grid_ridge = GridSearchCV(pipeline_ridge, param_grid=parametros_ridge, cv=5)
grid_ridge.fit(X_train, y_train)

y_pred_ridge = grid_ridge.predict(X_test)
mejor_alpha = grid_ridge.best_params_["regressor__alpha"]


### **Métricas del Modelo Ridge**

In [20]:
# Comparación entre MAE y R2

# Métricas para Regresión Lineal
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# Métricas para Ridge Regression
mae_ridge_model = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge_model = r2_score(y_test, y_pred_ridge)

print("\n--- Métricas Regresión Lineal ---")
print(f"{'R2':<6}: {r2_lr:.4f}")
print(f"{'MAE':<6}: {mae_lr:,.0f}")

print(f"\n--- Métricas del modelo Ridge (alpha={mejor_alpha:.1f}) ---")
print(f"{'R2':<6}: {r2_ridge_model:.4f}")
print(f"{'MAE':<6}: {mae_ridge_model:,.0f}")


--- Métricas Regresión Lineal ---
R2    : 0.5918
MAE   : 4,867

--- Métricas del modelo Ridge (alpha=1.0) ---
R2    : 0.5918
MAE   : 4,867


# ¿Cuál modelo presenta mejor desempeño?
Ambos modelos presentan exactamente el mismo desempeño ($R^2 = 0.5593$ y $MAE = 4.951$)

# ¿Las diferencias son significativas?
No son significativas. De hecho, el mejor parámetro de Ridge fue $\alpha = 0.1$. Al ser un valor tan cercano a 0, la penalización $L_2$ es casi nula, haciendo que el modelo Ridge se comporte de manera idéntica a la Regresión Lineal Múltiple.

### **Extracción y Análisis de Coeficientes del Modelo Lineal**

In [21]:
# 1. Aplicamos el preprocesador manualmente
X_limpio_matriz = preprocessor.fit_transform(X)

# 2. Convertimos el resultado de vuelta a un DataFrame para inspeccionarlo con Pandas
columnas_finales = preprocessor.get_feature_names_out()
X_limpio_df = pd.DataFrame(X_limpio_matriz, columns=columnas_finales)

# 3. Comprobar que ya no existen nulos en el set transformado
print("--- Cantidad de nulos después del preprocesador ---")
print(X_limpio_df.isnull().sum())

--- Cantidad de nulos después del preprocesador ---
num__tamano_tienda         0
num__antiguedad_tienda     0
num__temperatura           0
num__precio_combustible    0
num__indice_economico      0
num__semana                0
num__clientes_estimados    0
cat__tipo_tienda_B         0
cat__tipo_tienda_C         0
remainder__es_feriado      0
dtype: int64


In [22]:
X_limpio_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25006 entries, 0 to 25005
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   num__tamano_tienda       25006 non-null  float64
 1   num__antiguedad_tienda   25006 non-null  float64
 2   num__temperatura         25006 non-null  float64
 3   num__precio_combustible  25006 non-null  float64
 4   num__indice_economico    25006 non-null  float64
 5   num__semana              25006 non-null  float64
 6   num__clientes_estimados  25006 non-null  float64
 7   cat__tipo_tienda_B       25006 non-null  float64
 8   cat__tipo_tienda_C       25006 non-null  float64
 9   remainder__es_feriado    25006 non-null  float64
dtypes: float64(10)
memory usage: 1.9 MB


In [23]:
# Guardar datos limpios y transformados
columnas_finales = preprocessor.get_feature_names_out()
X_limpio_df = pd.DataFrame(
    preprocessor.fit_transform(X),
    columns=columnas_finales
)
X_limpio_df['ventas_semanales'] = y.values

import os
os.makedirs('../data/cleaned', exist_ok=True)
X_limpio_df.to_csv('../data/cleaned/retail_ventas_cleaned.csv', index=False)
print("Archivo guardado en ../data/cleaned/retail_ventas_cleaned.csv")

Archivo guardado en ../data/cleaned/retail_ventas_cleaned.csv


In [24]:
# Obtenemos los nombres de las columnas después de pasar por el ColumnTransformer
nombres_columnas = pipeline_lr.named_steps['preprocessor'].get_feature_names_out()

# Extraemos los pesos (coeficientes) del modelo entrenado
coeficientes = pipeline_lr.named_steps['regressor'].coef_

# Creamos un DataFrame para visualizar mejor el impacto de cada variable
importancia = pd.DataFrame({'Variable': nombres_columnas, 'Coeficiente': coeficientes})

# Ordenamos por valor absoluto para ver las más influyentes arriba
importancia = importancia.sort_values(by='Coeficiente', key=abs, ascending=False)

print("\n--- Importancia de las Variables Predictoras (Regresión Lineal) ---")
display(importancia)


--- Importancia de las Variables Predictoras (Regresión Lineal) ---


,Variable,Coeficiente
8,cat__tipo_tienda_C,-16286.817490
0,num__tamano_tienda,8641.836359
7,cat__tipo_tienda_B,-8241.431543
9,remainder__es_feriado,5289.088975
6,num__clientes_estimados,1623.512310
4,num__indice_economico,707.488442
3,num__precio_combustible,-70.595899
2,num__temperatura,-65.036675
1,num__antiguedad_tienda,-21.097498
5,num__semana,-4.731302


# **Interpretación de resultados**
### **Identificación de variables más influyentes:** ###



*   tipo_tienda
*   tamano_tienda
*   es_feriado
*   clientes_estimados



### **Explicación del efecto de 3 variables sobre las ventas:** ###

1.- *num__tamano_tienda* (Coef: $+8591.09$): Por cada incremento en una desviación estándar del tamaño de la tienda, las ventas semanales aumentan en promedio $\$8.591$, siendo la variable numérica más influyente.

2.- *num__clientes_estimados* (Coef: $+1620.44$): Un mayor volumen de clientes estimados impacta positivamente el rendimiento de las ventas semanales.

3.- *num__es_feriado* (Coef: $+1581.73$): Las semanas que contienen feriados generan un incremento promedio en las ventas de $\$1.581$, ideal para planificar stock y campañas promocionales.

### **Interpretación del impacto de la variable tipo_tienda:** ###

Al aplicar One-Hot Encoding eliminando la primera categoría (drop="first") , la Tienda A quedó como la categoría de referencia.  cat__tipo_tienda_C tiene un coeficiente de $-16193.40$. Esto significa que, a igualdad de condiciones, las tiendas pequeñas (Tipo C) venden en promedio $\$16.193$ menos que las tiendas grandes (Tipo A).  Las tiendas medianas (Tipo B) venden en promedio $\$8.113$ menos que las de Tipo A.  

# **Predicción**

In [25]:
nuevos_registros = pd.DataFrame([
    {
        'tipo_tienda': 'A', 'tamano_tienda': 180000, 'antiguedad_tienda': 12,
        'temperatura': 75.0, 'precio_combustible': 3.2, 'indice_economico': 0.5,
        'semana': 48, 'clientes_estimados': 8000,
        'es_feriado': 1   # ← semana 48 contiene feriado (ej. Navidad)
    },
    {
        'tipo_tienda': 'B', 'tamano_tienda': 90000, 'antiguedad_tienda': 5,
        'temperatura': 60.0, 'precio_combustible': 3.8, 'indice_economico': -0.3,
        'semana': 10, 'clientes_estimados': 3500,
        'es_feriado': 0   # ← semana normal
    },
    {
        'tipo_tienda': 'C', 'tamano_tienda': 35000, 'antiguedad_tienda': 2,
        'temperatura': 55.0, 'precio_combustible': 4.1, 'indice_economico': -1.0,
        'semana': 5, 'clientes_estimados': 1200,
        'es_feriado': 0   # ← semana normal
    }
])

predicciones = pipeline_lr.predict(nuevos_registros)

resultados = nuevos_registros.copy()
resultados['ventas_predichas'] = predicciones.round(0)
print(resultados[['tipo_tienda', 'tamano_tienda', 'clientes_estimados', 'es_feriado', 'ventas_predichas']])

  tipo_tienda  tamano_tienda  clientes_estimados  es_feriado  ventas_predichas
0           A         180000                8000           1         4034506.0
1           B          90000                3500           0         1999347.0
2           C          35000                1200           0          763404.0


# **Interpretación de Resultados**

Los resultados obtenidos son coherentes con el contexto de negocio.

* **Tienda tipo A ($4.034.506):** Registra las ventas predichas más altas, lo cual
es esperable dado su mayor tamaño (180.000 m²), alto flujo de clientes estimados
(8.000) y su ubicación en la semana 48 (período festivo, ej. Navidad), factor que
el modelo identifica como un impulsor relevante de la demanda.

* **Tienda tipo B ($1.999.347):** Presenta ventas moderadas, reflejo de un tamaño
intermedio (90.000 m²), tráfico regular de 3.500 clientes y condiciones económicas
levemente negativas (índice: -0.3), lo que modera su rendimiento sin afectarlo
de forma significativa.

* **Tienda tipo C ($763.404):** Obtiene las ventas más bajas, consistente con su
menor tamaño (35.000 m²), bajo flujo de clientes (1.200), semana sin feriado y el
índice económico más desfavorable (-1.0), que refleja menor poder adquisitivo
en su zona de influencia.



# **¿En qué situación preferiría utilizar Ridge en lugar de LinearRegression?**

Ridge es preferible cuando:

* Existe multicolinealidad entre predictores (ej. tamano_tienda y clientes_estimados pueden estar correlacionadas). Ridge estabiliza los coeficientes en ese caso.
* El modelo muestra sobreajuste: Ridge penaliza coeficientes grandes, reduciendo la varianza a costa de un ligero sesgo.
* Se busca estabilidad en los coeficientes ante pequeñas variaciones en los datos de entrenamiento.

En este caso, dado que LinearRegression y Ridge obtienen métricas casi idénticas, las diferencias son mínimas, lo que sugiere que no hay multicolinealidad severa ni sobreajuste significativo.